In [0]:
%pip install google-api-python-client google-auth google-auth-httplib2 --quiet
%pip install openpyxl

In [0]:
dbutils.library.restartPython()

In [0]:
from google.oauth2 import service_account
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
import io, os, re, tempfile, shutil
from collections import defaultdict
from pyspark.sql import functions as F

# --- Configurações ---
DRIVE_FOLDER_ID = "0AABpydeD9EpqUk9PVA"
CATALOG = "hering-data-business-prod"
SCHEMA = "ext_industria"

# --- Parâmetro para filtro seletivo de tabelas ---
dbutils.widgets.text("tabelas", "", "Tabelas a processar (separadas por vírgula)")

# --- Credenciais Google ---
import json

# Credenciais armazenadas no Volume do Unity Catalog
VOLUME_CRED_PATH = "/Volumes/hering-data-business-prod/ext_industria/credentials/hering_data_engine_prod_b505839675d9.json"

credentials = service_account.Credentials.from_service_account_file(
    VOLUME_CRED_PATH,
    scopes=["https://www.googleapis.com/auth/drive"]
)

print("Configurações e credenciais carregadas com sucesso.")

In [0]:
# Autenticação com Google Drive (usa credenciais da célula anterior)
drive_service = build("drive", "v3", credentials=credentials)

# Listar todos os arquivos na pasta do Drive
def list_drive_files(folder_id):
    """Lista recursivamente todos os arquivos em uma pasta do Google Drive."""
    all_files = []
    page_token = None
    query = f"'{folder_id}' in parents and trashed = false"
    
    while True:
        results = drive_service.files().list(
            q=query,
            spaces="drive",
            fields="nextPageToken, files(id, name, mimeType)",
            pageToken=page_token,
            supportsAllDrives=True,
            includeItemsFromAllDrives=True
        ).execute()
        
        files = results.get("files", [])
        for f in files:
            if f["mimeType"] == "application/vnd.google-apps.folder":
                # Recursar em subpastas
                all_files.extend(list_drive_files(f["id"]))
            else:
                all_files.append(f)
        
        page_token = results.get("nextPageToken")
        if not page_token:
            break
    
    return all_files

files = list_drive_files(DRIVE_FOLDER_ID)
print(f"Total de arquivos encontrados: {len(files)}")
for f in files:
    print(f"  - {f['name']} ({f['mimeType']})")

In [0]:
# Agrupar arquivos por nome base (removendo sufixo de ano)
def extract_base_and_year(filename):
    """Extrai o nome base e o ano do nome do arquivo.
    Ex: EBPAP02_2025.xlsx -> ('EBPAP02', '2025')
        EBPAP02_2026.csv  -> ('EBPAP02', '2026')
    """
    # Remover extensão
    name_no_ext = os.path.splitext(filename)[0]
    
    # Tentar extrair ano no padrão _YYYY no final
    match = re.match(r'^(.+?)_(\d{4})$', name_no_ext)
    if match:
        return match.group(1), match.group(2)
    
    # Se não encontrar padrão de ano, usar nome completo sem ano
    return name_no_ext, None

# Agrupar: {nome_base: [(file_info, ano), ...]}
file_groups = defaultdict(list)
for f in files:
    base_name, year = extract_base_and_year(f["name"])
    ext = os.path.splitext(f["name"])[1].lower()
    file_groups[base_name].append({
        "file_id": f["id"],
        "file_name": f["name"],
        "mime_type": f["mimeType"],
        "year": year,
        "extension": ext,
        "base_name": base_name
    })

print(f"Grupos de tabelas a serem criados: {len(file_groups)}")
for base, items in file_groups.items():
    years = [i['year'] for i in items if i['year']]
    print(f"  - {base}: {len(items)} arquivo(s) | Anos: {years if years else ['sem ano']}")

In [0]:
# Mapeamento de tipos Google Docs -> formato de exportação
GOOGLE_EXPORT_MAP = {
    "application/vnd.google-apps.spreadsheet": (
        "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet", ".xlsx"
    ),
    "application/vnd.google-apps.document": (
        "application/vnd.openxmlformats-officedocument.wordprocessingml.document", ".docx"
    ),
}

def download_file(file_id, file_name, dest_dir, mime_type=None):
    """Baixa um arquivo do Google Drive para o diretório local.
    Arquivos nativos do Google (Sheets, Docs) são exportados automaticamente."""
    export_info = GOOGLE_EXPORT_MAP.get(mime_type) if mime_type else None

    if export_info:
        export_mime, export_ext = export_info
        request = drive_service.files().export_media(fileId=file_id, mimeType=export_mime)
        # Garantir extensão correta no arquivo exportado
        base_name = os.path.splitext(file_name)[0]
        file_name = base_name + export_ext
    else:
        request = drive_service.files().get_media(fileId=file_id)

    dest_path = os.path.join(dest_dir, file_name)
    with open(dest_path, "wb") as fh:
        downloader = MediaIoBaseDownload(fh, request)
        done = False
        while not done:
            _, done = downloader.next_chunk()
    return dest_path


def read_file_to_df(file_path, extension, skiprows=0):
    """Lê um arquivo local e retorna um Spark DataFrame.
    Para arquivos Excel, lê somente a 1ª aba (sheet_name=0).
    
    Args:
        file_path: Caminho do arquivo local.
        extension: Extensão do arquivo.
        skiprows: Número de linhas iniciais a pular (padrão: 0).
    """
    import pandas as pd
    
    ext = extension.lower()
    
    if ext in [".xlsx", ".xls"]:
        pdf = pd.read_excel(file_path, sheet_name=0, dtype=str, skiprows=skiprows)
        return spark.createDataFrame(pdf)
    elif ext == ".csv":
        if skiprows > 0:
            pdf = pd.read_csv(file_path, dtype=str, skiprows=skiprows)
            return spark.createDataFrame(pdf)
        return spark.read.option("header", "true").option("inferSchema", "false").csv(f"file:{file_path}")
    elif ext == ".parquet":
        return spark.read.parquet(f"file:{file_path}")
    elif ext == ".json":
        return spark.read.json(f"file:{file_path}")
    elif ext == ".txt" or ext == ".tsv":
        return spark.read.option("header", "true").option("delimiter", "\t").option("inferSchema", "false").csv(f"file:{file_path}")
    else:
        raise ValueError(f"Extensão não suportada: {ext}")

print("Funções de download e leitura definidas.")

In [0]:
# Configuração de tratamento especial por arquivo
# skiprows: número de linhas iniciais a pular
# columns: lista de colunas a importar (None = todas)
# date_columns: lista de colunas a converter para tipo date
# drop_duplicates: se True, remove linhas duplicadas
SPECIAL_FILES = {
    "FATO_PREVISAO_ENTREGA_ORDENS": {
        "skiprows": 2,
        "columns": ["ORDEM", "PRAZO"],
        "date_columns": ["PRAZO"],
        "drop_duplicates": True
    },
    "FATO_METAS_CC": {
        "skiprows": [1, 2],  # Pular linhas 2 e 3 do relatório (linha vazia e sub-cabeçalho "DIA SEMANA")
    }
}

# --- Filtro seletivo de tabelas ---
tabelas_param = dbutils.widgets.get("tabelas").strip()
if tabelas_param:
    tabelas_selecionadas = [t.strip().upper() for t in tabelas_param.split(",")]
    file_groups = {k: v for k, v in file_groups.items() if k in tabelas_selecionadas}
    print(f"Filtro ativo: processando apenas {list(file_groups.keys())}")
else:
    print("Sem filtro: processando todas as tabelas")

# Processar cada grupo de arquivos e criar tabelas no Unity Catalog
tmp_dir = tempfile.mkdtemp(prefix="gdrive_")
success_count = 0
error_files = []

def sanitize_column_name(col_name):
    """Remove caracteres inválidos dos nomes de colunas."""
    sanitized = re.sub(r'[ ,;{}()\n\t=]+', '_', col_name)
    sanitized = sanitized.strip('_')
    sanitized = re.sub(r'_+', '_', sanitized)
    return sanitized

def auto_cast_columns(df, sample_size=1000):
    """Detecta e converte automaticamente os tipos das colunas (DATE, INTEGER, DOUBLE).
    Usa try_cast para tratar valores inválidos como NULL."""
    import re as _re
    
    date_patterns = [
        r'^\d{4}-\d{2}-\d{2}$',             # 2026-01-15
        r'^\d{2}/\d{2}/\d{4}$',             # 15/01/2026
        r'^\d{4}-\d{2}-\d{2}T',             # 2026-01-15T...
        r'^\d{4}-\d{2}-\d{2} \d{2}:\d{2}',  # 2026-01-15 10:30
    ]
    
    casts_applied = []
    
    for col_name in df.columns:
        sample_rows = (
            df.filter(F.col(f"`{col_name}`").isNotNull() & (F.trim(F.col(f"`{col_name}`").cast("string")) != ""))
            .select(F.col(f"`{col_name}`").cast("string").alias("val"))
            .limit(sample_size)
            .collect()
        )
        
        if not sample_rows:
            continue
        
        values = [row["val"] for row in sample_rows if row["val"] is not None]
        if not values:
            continue
        
        # --- Tentar DATE ---
        is_date = False
        for pattern in date_patterns:
            matches = sum(1 for v in values if _re.match(pattern, v.strip()))
            if matches / len(values) >= 0.8:
                is_date = True
                break
        
        if is_date:
            # Usar try_cast para tolerar valores malformados (retorna NULL)
            df = df.withColumn(col_name, F.expr(f"try_cast(`{col_name}` AS DATE)"))
            casts_applied.append((col_name, "DATE"))
            continue
        
        # --- Tentar INTEGER ---
        numeric_values = [v.strip().replace(",", "") for v in values if v.strip() not in ("-", "N/A", "n/a", "")]
        
        if numeric_values:
            int_matches = 0
            for v in numeric_values:
                try:
                    num = float(v)
                    if num == int(num) and abs(num) < 2**31:
                        int_matches += 1
                except (ValueError, OverflowError):
                    pass
            
            if int_matches / len(numeric_values) >= 0.8:
                df = df.withColumn(col_name, F.expr(f"CAST(try_cast(`{col_name}` AS DOUBLE) AS INT)"))
                casts_applied.append((col_name, "INTEGER"))
                continue
            
            # --- Tentar DOUBLE ---
            double_matches = 0
            for v in numeric_values:
                try:
                    float(v)
                    double_matches += 1
                except ValueError:
                    pass
            
            if double_matches / len(numeric_values) >= 0.8:
                df = df.withColumn(col_name, F.expr(f"try_cast(`{col_name}` AS DOUBLE)"))
                casts_applied.append((col_name, "DOUBLE"))
                continue
    
    return df, casts_applied

try:
    for base_name, file_list in file_groups.items():
        print(f"\n{'='*60}")
        print(f"Processando grupo: {base_name} ({len(file_list)} arquivo(s))")
        
        # Verificar se há tratamento especial para este grupo
        special = SPECIAL_FILES.get(base_name, {})
        skiprows = special.get("skiprows", 0)
        select_columns = special.get("columns", None)
        date_columns = special.get("date_columns", [])
        drop_duplicates = special.get("drop_duplicates", False)
        
        if special:
            print(f"  [Tratamento especial] skiprows={skiprows}, columns={select_columns}, date_columns={date_columns}, drop_duplicates={drop_duplicates}")
        
        dfs = []
        for file_info in file_list:
            try:
                # Download (passa mime_type para tratar Google Sheets nativos)
                print(f"  Baixando: {file_info['file_name']}...")
                local_path = download_file(
                    file_info["file_id"], 
                    file_info["file_name"], 
                    tmp_dir,
                    file_info["mime_type"]
                )
                
                # Usar extensão do arquivo baixado (pode diferir do original após export)
                actual_ext = os.path.splitext(local_path)[1].lower()
                ext_to_use = actual_ext if actual_ext else file_info["extension"]
                
                # Ler para DataFrame (com skiprows se configurado)
                df = read_file_to_df(local_path, ext_to_use, skiprows=skiprows)
                
                # Sanitizar nomes de colunas
                for old_col in df.columns:
                    new_col = sanitize_column_name(old_col)
                    if new_col != old_col:
                        df = df.withColumnRenamed(old_col, new_col)
                
                # Selecionar apenas colunas específicas se configurado
                if select_columns:
                    sanitized_select = [sanitize_column_name(c) for c in select_columns]
                    available_cols = [c for c in sanitized_select if c in df.columns]
                    missing_cols = [c for c in sanitized_select if c not in df.columns]
                    if missing_cols:
                        print(f"    AVISO: Colunas não encontradas: {missing_cols}")
                    if available_cols:
                        df = df.select(*available_cols)
                    else:
                        print(f"    ERRO: Nenhuma coluna solicitada encontrada. Colunas disponíveis: {df.columns}")
                        continue
                
                # Converter colunas para tipo date se configurado
                for col_name in date_columns:
                    sanitized_col = sanitize_column_name(col_name)
                    if sanitized_col in df.columns:
                        df = df.withColumn(sanitized_col, F.to_date(F.col(sanitized_col)))
                        print(f"    Coluna '{sanitized_col}' convertida para tipo date")
                
                # Remover linhas duplicadas se configurado
                if drop_duplicates:
                    count_before = df.count()
                    df = df.dropDuplicates()
                    count_after = df.count()
                    print(f"    Duplicatas removidas: {count_before - count_after} linhas ({count_before} -> {count_after})")
                
                # Adicionar coluna de ano somente se o arquivo tiver sufixo de ano
                if file_info["year"]:
                    df = df.withColumn("ano", F.lit(file_info["year"]))
                
                dfs.append(df)
                print(f"    -> {df.count()} linhas lidas")
                
                # Limpar arquivo local
                os.remove(local_path)
                
            except Exception as e:
                print(f"  ERRO ao processar {file_info['file_name']}: {e}")
                error_files.append((file_info["file_name"], str(e)))
        
        if not dfs:
            print(f"  Nenhum DataFrame criado para {base_name}. Pulando.")
            continue
        
        # Unir todos os DataFrames do grupo (unionByName com allowMissingColumns)
        combined_df = dfs[0]
        for df in dfs[1:]:
            combined_df = combined_df.unionByName(df, allowMissingColumns=True)
        
        # Ajuste automático de tipos (DATE, INTEGER, DOUBLE)
        combined_df, casts = auto_cast_columns(combined_df)
        if casts:
            for col_name, tipo in casts:
                print(f"  🔄 Coluna '{col_name}' → {tipo}")
        else:
            print(f"  ℹ️  Nenhuma conversão de tipo aplicada")
        
        # Sanitizar nome da tabela (remover caracteres inválidos)
        table_name = re.sub(r'[^a-zA-Z0-9_]', '_', base_name).lower()
        full_table_name = f"`{CATALOG}`.`{SCHEMA}`.`{table_name}`"
        
        print(f"  Escrevendo tabela: {full_table_name} ({combined_df.count()} linhas totais, {len(combined_df.columns)} colunas)")
        
        combined_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(full_table_name)
        
        success_count += 1
        print(f"  ✅ Tabela {full_table_name} criada com sucesso!")

finally:
    # Limpar diretório temporário
    shutil.rmtree(tmp_dir, ignore_errors=True)

print(f"\n{'='*60}")
print(f"Processamento concluído!")
print(f"  Tabelas criadas com sucesso: {success_count}")
print(f"  Arquivos com erro: {len(error_files)}")
if error_files:
    print("\nArquivos com erro:")
    for fname, err in error_files:
        print(f"  - {fname}: {err}")